In [ ]:
%xmode minimal

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

jax.config.update('jax_enable_x64', True)

# While loops in JAX

In general, a Python `while` loop on a traced value can't compile.
`jax.lax.while_loop` covers this case case ([link to documentation](https://docs.jax.dev/en/latest/_autosummary/jax.lax.while_loop.html), read it!).
It loops until a data-dependent condition turns false, even when the step count isn't known in advance.

You give it a condition, a body that returns the updated loop state, and an initial state, and it threads that state through each iteration (its shape and dtype must stay fixed).

In this notebook, we will explore some concrete example using `jax.lax.while_loop` and also take a look at `jax.lax.fori_loop`.

## Example: fixed-point iterations

We want to solve equations of the type
$$f(x)=x \leftrightarrow f(x)-x=0,$$
which is equivalent to finding the root(s) of $g(x)=f(x)-x$.

Let
$$f(x)=\sin x+x.$$
It has fixed points every $n\pi, n \in \mathbb{Z}$, that is, $f(n\pi)=n\pi$.

We will explore this function and fixed-point iterations throughout this section as a concret example to illustrate the looping functions in JAX.

We first plot $f$ and $y=x$. Indeed, there seems to an intersection between both curves every multiple of $\pi$.

In [ ]:
def f(x):
    return jnp.sin(x)+x
    
x = jnp.linspace(0., 10., 100)

plt.figure(figsize=(5., 5.), layout='constrained')
plt.plot(x, f(x), label="$y=\\sin(x)+x$")
plt.plot(x, x, label="$y=x$")
plt.grid()
plt.legend()
plt.xlabel("$x$")
plt.ylabel("$y$")
plt.show()

### First naive implementation
We first implement the fixed point algorithm naively using eager mode JAX (as if using NumPy).

Below is a basic implementation illustrating the fixed point algorithm and some evaluations of it at different points on the abscisses.

In [ ]:
def fixed_point_naive(f, x_init):
    x = f(x_init)
    while not jnp.isclose(x, x_init):
        x_init = x
        x = f(x)
    return x

print(fixed_point_naive(f, 2.))
print(fixed_point_naive(f, 6.))
print(fixed_point_naive(f, 1.))
print(fixed_point_naive(f, 8.))

We see that indeed it does converges to integer multiples of $\pi$.
However, it seems that it only does so to odd multiples. We will explore this later.

Now we try to jit this function using `jax.jit` (specifying the `f` parameter as a static argument, since functions are not Pytrees).

In [ ]:
jitted_fixed_point_naive = jax.jit(fixed_point_naive, static_argnames=['f'])
print(jitted_fixed_point_naive(f, 2.))

Doing so fails. Indeed, the Python `while` operator will terminate depending on the initial $x$ value.
And, as we saw in the basic JAX notebook, control flow can't depend on input values.

### Fixed looping - Unrolled while loop

One way to counter this problem is to loop only a set amount of time.

By setting the loop count $N$ as a static argument, the flow of the function is the same between each execution for a given $N$.
However, different loop counts will retrigger a compilation.
In addition, the computation in the while loop gets unrolled, which can lead to (very) long compilation time and excessive binary size.

In [ ]:
@jax.jit(static_argnames=('f', 'N'))
def fixed_point_naive_fix(f, x_init, N):
    print("Compiling fixed_point_naive_fix(), N={}...".format(N))
    x = f(x_init)
    i = 0
    while i < N:
        x = f(x)
        i += 1
    return x

In [ ]:
fixed_point_naive_fix(f, 3., 10)
fixed_point_naive_fix(f, 3., 10)
fixed_point_naive_fix(f, 3., 10)
fixed_point_naive_fix(f, 3., 20)

As seen, for 2 different $N$, the function is compiled twice.

> _**Question:**_ What happens when you put a very high number of loops? (like, $N=10^7$). Add another print after the while loop to show this.

A way to counter the penality of recompilation (and avoid excessive unrolling) is to use the `jax.lax.fori_loop` function, presented below.

### Fixed looping using `jax.lax.fori_loop`
One way to avoid the unrolling and recompilation issues is to use the `jax.lax.fori_loop` function ([link to documentation](https://docs.jax.dev/en/latest/_autosummary/jax.lax.fori_loop.html), read it!).

In [ ]:
@jax.jit(static_argnames=('f',))
def fixed_point_fori_loop(f, x_init, N):
    print("Compiling fixed_point_fori_loop()...")
    lower = 0
    upper = N
    def body_fun(i, x):
        return f(x)

    return jax.lax.fori_loop(lower, upper, body_fun, x_init)

In [ ]:
fixed_point_fori_loop(f, 2., 10)
fixed_point_fori_loop(f, 2., 20)

> _**Question:**_ Same question as for the naive `while` loop, put a very large N and see what happens. What changed between last implemention?

> _**Question:**_ What happens when we define `N` as a static argument to `jit`?

### Using `jax.lax.while_loop`
While `jax.lax.fori_loop` fixed the recompilation issues, it does howevert divert us from our original algorithm, that is a fixed point iteration scheme which only stops once convergence is achieved.

To solve this problem, the proper function to use is `jax.lax.while_loop` ([link to documentation](https://docs.jax.dev/en/latest/_autosummary/jax.lax.while_loop.html), read it!).
As with many functions from the `lax` submodule, it is a primitive which directly maps into a XLA operation.

> _**NOTE:**_ Actually, `jax.lax.fori_loop` converts back into a `jax.lax.while_loop` when the iteration count is not known at compile time.

> _**NOTE:**_ The price to pay with `while_loop` is that it is not reverse-mode differentiable.
> We will see later on, when exploring the JAX autograd engine, why it (might) matters.

Rewriting the algorithm using `while_loop` is done as follows:

In [ ]:
@jax.jit(static_argnames=('f',))
def fixed_point_lax_while_loop(f, x_init, atol=1e-8, rtol=1e-5):
    print("Compiling fixed_point_lax_while_loop()...")
    def _cond_fun(x):
        return jnp.logical_not(jnp.isclose(x[0], x[1], rtol=rtol, atol=atol))

    def _body_fun(x):
        return (x[1], f(x[1]))
        
    return jax.lax.while_loop(_cond_fun,
                              _body_fun,
                              (0., f(x_init)))[1]


In [ ]:
print(fixed_point_lax_while_loop(f, 3.))
print(fixed_point_lax_while_loop(f, 6.))
print(fixed_point_lax_while_loop(f, 8.))

### Exercise - Mapping the attractor basins
As we saw early on, the fixed point algorithm seems to converge only on odd multiple of $\pi$.

To show this, using `jax.vmap`, evaluate the fixed point algorithm on many initial positions and plot them against their converged value.
Also plot the derivate of $f$. What do you see? Also evaluate at even multiples of $\pi$

In [ ]:
x_init = jnp.linspace(0., 10., 10000)
y = jax.vmap(lambda x_init: fixed_point_lax_while_loop(f, x_init))(x_init)

In [ ]:
plt.plot(x_init, y)
plt.plot(x, f(x))
plt.plot(x, jnp.cos(x)+1)
plt.grid()
plt.show()

## Exercice - Newton-Raphson algorithm and fractals

To find the roots of a given function numerically (when there is no closed form solution for example), a common algorithm called Newton-Raphson can be used. It is a direct application of fixed point theory.

Given a current guess $x_n$, one can linearize $f$ around it
$$f(x) \approx f(x_n) + (x-x_n)f'(x_n).$$

Analiticaly solving its root gets us to the next guess,
$$x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}.$$

Repeating the operation (hopefuly) converges to a root.
However, as we saw before, a given root can be an attraction basin or repulsive.

In this exercice, we will explore the [complex structure](https://en.wikipedia.org/wiki/Newton_fractal) that can arise when applying numerical root finding algorithms to complex functions.

Let
$$
\begin{align}
f: & \mathbb{C} & \longrightarrow & \mathbb{C} \\
   & z & \longmapsto & f(z) = z^3-1
\end{align}.
$$

It can be shown that this function has 3 fixed points, [namely the 3 cube roots of unity](https://en.wikipedia.org/wiki/Cube_root):
$
\begin{Bmatrix}
 1, \frac{-1+i\sqrt{3}}{2}, \frac{-1-i\sqrt{3}}{2}
\end{Bmatrix}.
$

We will show this graphically.

First, modify the previous fixed-point algorithm to also output the step count up to convergence.

In [ ]:
# Uncomment to get the solution!
# %load solutions/loops_modified_fp.py

We now define the mapping on which we will apply the Newton-Raphson algorithm on, that is
$$g(z) = z - \frac{f(z)}{f'(z)} = z - \frac{z^3-1}{3z^2}.$$

We then define a grid on the complex plane. 

In [ ]:
def f(z):
    return z**3-1

def df(z):
    return 3*z**2
    
def g(z):
    return z-f(z)/df(z)

N = 1000
XY = jnp.stack(jnp.meshgrid(jnp.linspace(-2., 2., N), jnp.linspace(-2., 2., N))).reshape(2, -1).T
z0 = XY[:, 0]+XY[:, 1]*1j

We now vmap the fixed point algorithm on each points of the complex plane defined above:

In [ ]:
Z = jax.vmap(fixed_point, in_axes=(None, 0))(g, z0)

We can now identify in which attraction basin each initial pixel position went and color code them. We then plot.

In [ ]:

C = 0.*jnp.isclose(Z[0], (-1+1j*jnp.sqrt(3))/2) + \
    1.*jnp.isclose(Z[0], (-1-1j*jnp.sqrt(3))/2) + \
    2.*jnp.isclose(Z[0], 1.)


plt.subplots(ncols=2, nrows=1, figsize=(8., 4.), layout='constrained')
plt.subplot(1, 2, 1)
plt.title("Attraction basin")
plt.imshow(C.reshape(N, N))
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title("Iteration count")
plt.imshow(Z[1].reshape(N, N))
plt.axis('off')

plt.show()

## Application - Collatz steps

One more application if you are up to it (feel free to pass to the next notebook).

This time, we will try to solve the [Collatz conjecture](https://en.wikipedia.org/wiki/Collatz_conjecture) (and get a nice medal for it).

Map the Collatz step over a wide array of integers (lets say up to $10^6$) and plot its total step count against its initial value.

In [ ]:
# Uncomment to get the solution!
# %load solutions/loops_collatz.py